# Week 6 · Day 4 — SQL **and** Python, working together

*Let the warehouse do the heavy lifting; let Python do the last mile — charts, and the hand-off to Claude.*

**By the end you'll have shipped:** an end-to-end mini-pipeline — **SQL aggregates in the warehouse → pandas DataFrame → a matplotlib chart → a Claude-ready summary** — the pattern behind every tool you'll build from here on.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1b · SQL foundations → M2 · Building with Claude (Week 6) |
| **Prerequisites** | W6D1–D3 (SELECT / GROUP BY / JOIN), Week 2 pandas |
| **Est. time** | ~30 min |
| **Capstone slice** | The **spine** of *Matter Intelligence*: query in SQL → shape in Python → feed Claude |
| **Difficulty** | Core + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — DuckDB + a quick `matplotlib` install (auto-handled), no key needed |

### 🎯 Learning objectives

By the end you'll be able to:
- Explain the **division of labor**: what belongs in SQL vs. what belongs in Python.
- Run a SQL aggregate and **chart the resulting DataFrame** with matplotlib.
- **Parameterize** a query safely from Python variables (and know the injection risk).
- **Persist** a query result back into the warehouse as a new table.
- Package the whole thing into one reusable function — and tee the result up for **Claude**.

### ⚖️ Why it matters

SQL and pandas aren't rivals — they're a relay team. **SQL** is unbeatable at filtering, joining, and aggregating **huge** tables *where the data lives* (it never leaves the warehouse). **Python** is unbeatable at the **last mile**: reshaping, plotting, and handing data to an LLM. Every tool you'll build — including *Matter Intelligence* — is exactly this handoff: *query in SQL, finish in Python.*

### ⚙️ Setup

Loads `coffee_orders` via the usual `run_sql` helper, **plus** matplotlib for charts (installed automatically if missing).

> 🔒 *Synthetic data only — this coffee set (and the matters set) is fake. Never load real client or privileged data into a teaching notebook.*

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Load .env if python-dotenv is present (optional — the notebook runs fine without it).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# --- Pick the backend: Snowflake if credentials exist in .env, else local DuckDB ---
# You write the SAME SQL either way; the backend is invisible.
SNOWFLAKE_READY = all(os.environ.get(k) for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"))
BACKEND = "snowflake" if SNOWFLAKE_READY else "duckdb"

def _find(fname):
    for base in ("../../data/", "data/", ""):
        if os.path.exists(base + fname):
            return base + fname
    return None

# Tables this lesson needs — loaded from Training/data/, with a tiny built-in fallback.
FALLBACK = {
    'coffee_orders': [{'order_id': 'O-5001', 'date': '2026-03-07', 'item': 'Cappuccino', 'size': 'S', 'category': 'Espresso Drink', 'price': 3.75, 'payment': 'Cash', 'store': 'Downtown'}, {'order_id': 'O-5002', 'date': '2026-03-07', 'item': 'Mocha', 'size': 'S', 'category': 'Espresso Drink', 'price': 4.5, 'payment': 'Card', 'store': 'Airport'}, {'order_id': 'O-5003', 'date': '2026-03-05', 'item': 'Cappuccino', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.25, 'payment': 'Card', 'store': 'Downtown'}, {'order_id': 'O-5004', 'date': '2026-03-05', 'item': 'Croissant', 'size': 'M', 'category': 'Food', 'price': 3.25, 'payment': 'App', 'store': 'Uptown'}, {'order_id': 'O-5005', 'date': '2026-03-06', 'item': 'Latte', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.5, 'payment': 'App', 'store': 'Downtown'}],
}
frames = {}
for _name, _rows in FALLBACK.items():
    _p = _find(_name + ".csv")
    frames[_name] = pd.read_csv(_p) if _p else pd.DataFrame(_rows)

if BACKEND == "duckdb":
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
        import duckdb
    _con = duckdb.connect(":memory:")                 # a private, in-memory warehouse
    for _name, _df in frames.items():
        _con.register("_src_" + _name, _df)
        _con.execute(f"CREATE OR REPLACE TABLE {_name} AS SELECT * FROM _src_{_name}")
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        return _con.execute(sql).df()
else:
    import snowflake.connector
    from snowflake.connector.pandas_tools import write_pandas
    _con = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"], warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ.get("SNOWFLAKE_DATABASE"), schema=os.environ.get("SNOWFLAKE_SCHEMA"))
    for _name, _df in frames.items():
        write_pandas(_con, _df, _name.upper(), auto_create_table=True, overwrite=True, quote_identifiers=False)
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        cur = _con.cursor(); cur.execute(sql); return cur.fetch_pandas_all()

print(f"✅ Ready. Backend = {BACKEND.upper()} · tables: {', '.join(frames)}")

In [ ]:
# matplotlib powers the chart in Section 2 (auto-install if it's not here yet).
try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
    import matplotlib
import matplotlib.pyplot as plt
print(f"✅ matplotlib {matplotlib.__version__} ready")

### 1 · The handoff — `run_sql` returns a pandas DataFrame

You've used this all week: `run_sql(...)` sends SQL to the warehouse and returns a **pandas DataFrame**. That return value is the seam between the two worlds — SQL on the left, Python on the right.

In [ ]:
# SQL does the aggregation (in the warehouse); pandas receives a tidy result.
by_cat = run_sql("""
    SELECT category, ROUND(SUM(price), 2) AS revenue
    FROM coffee_orders
    GROUP BY category
    ORDER BY revenue DESC
""")
print(type(by_cat))          # <class 'pandas.core.frame.DataFrame'>
by_cat

### 2 · Chart it — SQL aggregates, matplotlib draws

Once it's a DataFrame, every pandas/plotting tool applies. Here SQL computed revenue-per-category; matplotlib turns those few rows into a bar chart. **Key idea:** we aggregated *first* in SQL, so Python only ever handles the small summary — not millions of raw rows.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(by_cat["category"], by_cat["revenue"], color="#4C78A8")
ax.set_title("Revenue by category")
ax.set_ylabel("Revenue ($)")
ax.set_xlabel("")
for i, v in enumerate(by_cat["revenue"]):
    ax.text(i, v + 0.3, f"${v:,.0f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

**What just happened:** the warehouse did the `GROUP BY`/`SUM`; Python did the drawing. That split is the whole point — **push aggregation down to SQL, keep presentation in Python.**

### 3 · Parameterize a query from Python

Often the filter value is a Python variable (a store the user picked, a date range). You can build the SQL with an f-string — but the **safe** way to inject *values* is a **bound parameter** (`?`), which the engine escapes for you.

In [ ]:
chosen_store = "Downtown"

# ✅ Safer: pass the value as a bound parameter, not glued into the string.
q = "SELECT order_id, item, price FROM coffee_orders WHERE store = ? ORDER BY price DESC"
try:
    result = _con.execute(q, [chosen_store]).df()        # DuckDB binds `?`
except Exception:
    result = run_sql(f"SELECT order_id, item, price FROM coffee_orders WHERE store = '{chosen_store}' ORDER BY price DESC")
print(f"Top orders at {chosen_store}:")
result.head()

> **`Common pitfalls ⚠️` — SQL injection.** Gluing raw user text into SQL (`f"... = '{user_text}'"`) lets a malicious value like `x' OR '1'='1` rewrite your query. For *values*, prefer **bound parameters** (`?` / `%s`). f-strings are fine for values *you* control (like a hard-coded column name), but never for untrusted input. We'll treat this properly when we build the API in the backend module.

### 4 · Persist a result back to the warehouse

A query result is often worth **saving** — a nightly summary table others can query. `CREATE TABLE AS SELECT` (**CTAS**) stores a query's output as a new table, in one statement.

In [ ]:
# Materialize the category summary as its own table, then read it back.
run_sql("CREATE OR REPLACE TABLE category_summary AS "
        "SELECT category, ROUND(SUM(price),2) AS revenue, COUNT(*) AS orders "
        "FROM coffee_orders GROUP BY category")

run_sql("SELECT * FROM category_summary ORDER BY revenue DESC")

**What just happened:** `CREATE TABLE AS SELECT` ran the aggregation and stored the result as `category_summary`. Now any teammate (or dashboard) can `SELECT * FROM category_summary` without recomputing it. In Snowflake this is how you build the curated tables your reports sit on — and, next lesson, how you load and shape data at scale.

> **`Go Deeper 🔧` — when to use SQL vs. pandas.**
>
> | Do it in **SQL** (in the warehouse) | Do it in **Python/pandas** (after) |
> |---|---|
> | Filter/aggregate **big** tables | Plot, style, export |
> | Joins across large tables | Row-by-row custom logic |
> | Anything that shrinks the data | Feed results to an LLM / API |
> | Work shared by many queries (CTAS) | One-off exploration |
>
> **Rule of thumb:** *make the data small in SQL first, then bring it to Python.* Never `SELECT *` a huge table into pandas just to filter it there.

### ✍️ Your turn

In [ ]:
# Delete the `pass` lines and write your code.

# TODO 1: query average price per STORE into a DataFrame called `by_store`
#         (SELECT store, ROUND(AVG(price),2) AS avg_price ... GROUP BY store)
pass

# TODO 2: draw a bar chart of by_store  (ax.bar(by_store["store"], by_store["avg_price"]))
pass

# TODO 3: save a `store_summary` table with revenue + order count per store (CTAS)
pass


<details><summary>✅ Show solution</summary>

```python
# 1
by_store = run_sql("""
    SELECT store, ROUND(AVG(price), 2) AS avg_price
    FROM coffee_orders
    GROUP BY store
    ORDER BY avg_price DESC
""")

# 2
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(by_store["store"], by_store["avg_price"], color="#F58518")
ax.set_title("Average order value by store"); ax.set_ylabel("Avg price ($)")
plt.tight_layout(); plt.show()

# 3
run_sql("CREATE OR REPLACE TABLE store_summary AS "
        "SELECT store, ROUND(SUM(price),2) AS revenue, COUNT(*) AS orders "
        "FROM coffee_orders GROUP BY store")
run_sql("SELECT * FROM store_summary ORDER BY revenue DESC")
```
</details>

### 🚀 Build the artifact — a query → chart → Claude-ready pipeline

One function that runs the whole relay: **SQL** aggregates, **pandas** receives, **matplotlib** charts, and we assemble a compact **text summary** — the exact input you'd hand Claude next module to write the narrative ("Espresso Drinks led at \$X...").

In [ ]:
def category_briefing():
    """SQL aggregate -> DataFrame -> chart -> a text summary ready for an LLM."""
    # 1. SQL: aggregate in the warehouse
    df = run_sql("""
        SELECT category, ROUND(SUM(price),2) AS revenue, COUNT(*) AS orders
        FROM coffee_orders GROUP BY category ORDER BY revenue DESC
    """)

    # 2. Python: chart it
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(df["category"][::-1], df["revenue"][::-1], color="#54A24B")
    ax.set_title("Revenue by category"); ax.set_xlabel("Revenue ($)")
    plt.tight_layout(); plt.show()

    # 3. Python: build a compact briefing (this is what we'll hand to Claude in M2)
    lines = [f"- {r.category}: ${r.revenue:,.2f} across {r.orders} orders" for r in df.itertuples()]
    briefing = "Coffee sales by category:\n" + "\n".join(lines)
    return df, briefing

df, briefing = category_briefing()
print(briefing)
print("\n\u2192 Next module: send this `briefing` to Claude to draft the narrative summary.")

> **🔗 Your world — from coffee to matters.** This pipeline is *Matter Intelligence* in miniature. Swap the query:
>
> ```sql
> SELECT practice_area, ROUND(SUM(amount_billed),2) AS billed, COUNT(*) AS matters
> FROM matters WHERE status = 'Active' GROUP BY practice_area ORDER BY billed DESC;
> ```
>
> …chart it for a partner deck, then hand the same compact `briefing` text to Claude to write *"Litigation led active billings at \$X across N matters..."*. **SQL finds the numbers; Claude writes the story.** That's the whole product.

### 📝 Recap — what you shipped

- SQL and Python are a **relay**: aggregate/join **in SQL**, then shape/plot/LLM **in Python**.
- `run_sql(...)` returns a **DataFrame** — the seam between warehouse and Python.
- **Chart** a result with matplotlib; **parameterize** queries with bound parameters (`?`) to avoid injection.
- **Persist** results with `CREATE TABLE AS SELECT` so others can reuse them.
- **Artifact:** `category_briefing()` — a query → chart → Claude-ready summary, the spine of the capstone.

### 🧠 Check your understanding

1. Why aggregate **in SQL** before bringing data into pandas, instead of `SELECT *` and grouping in Python?
2. What's the risk of building a `WHERE` clause with an f-string and untrusted input — and the safer alternative?
3. What does `CREATE TABLE AS SELECT` do?
4. In the capstone pipeline, which tool finds the numbers and which writes the narrative?

<details><summary>✅ Answers</summary>

1. The warehouse aggregates close to the data and returns only the **small** summary — moving millions of raw rows to your laptop is slow and often impossible.
2. **SQL injection** — a crafted value can rewrite the query. Use **bound parameters** (`?` / `%s`) for values; reserve f-strings for values you control.
3. It runs the `SELECT` and stores its result as a **new table** in one statement (materializing a summary).
4. **SQL** finds the numbers; **Claude** writes the narrative.
</details>

### ➡️ Next up — Week 7, Day 1: Snowflake building blocks

Week 6 you *queried* tables. Week 7 you **build and run** them like a real warehouse: the account → warehouse → database → schema → table hierarchy, `CREATE TABLE`, `INSERT`, data types — and you'll stand up your own **`matters`** table from scratch.

*Same toolkit, no install needed.*

### 📖 Reference & glossary

| Term | Plain meaning |
|---|---|
| **Division of labor** | SQL aggregates/joins at scale; Python shapes, plots, and calls the LLM |
| **Bound parameter (`?`)** | a placeholder the engine safely substitutes a value into (prevents injection) |
| **SQL injection** | attack where untrusted text rewrites your query |
| **`CREATE TABLE AS SELECT` (CTAS)** | store a query's result as a new table |
| **Briefing** | a compact text summary of query results, ready to hand an LLM |

**Docs:** matplotlib — https://matplotlib.org/stable/ · Snowflake CTAS — https://docs.snowflake.com/en/sql-reference/sql/create-table

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI or data output that will be relied upon.*